In [1]:
"""
Fixed CLEVR Dataset - Exploratory Data Analysis
Compatible with the actual dataset implementation
"""

import json
import sys
from pathlib import Path
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from PIL import Image

# Determine root of the repo
try:
    root_dir = Path(__file__).resolve().parent.parent
except NameError:
    root_dir = Path('.').resolve().parent

# Add 'src' to sys.path
sys.path.append(str(root_dir / 'src'))

# Import dataset loader
from data_loaders.clevrDataset import CLEVRDataset

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)


class CLEVREDAAnalyzer:
    """Comprehensive EDA for CLEVR dataset"""
    
    def __init__(self, data_dir: str, split: str = 'train'):
        self.data_dir = Path(data_dir)
        self.split = split
        
        # Load data
        print(f"Loading {split} split...")
        self.dataset = CLEVRDataset(data_dir, split=split, max_samples=None)

        # Load raw JSON for detailed analysis
        questions_path = self.data_dir / f"processed_{split}_questions.jsonl"
        self.questions_json = []
        with open(questions_path, 'r') as f:
            for line in f:
                self.questions_json.append(json.loads(line))

        # Load scenes (not available for test split)
        self.scenes_json = None
        if split != 'test':
            scenes_path = self.data_dir / f"processed_{split}_scenes.jsonl"
            self.scenes_json = []
            with open(scenes_path, 'r') as f:
                for line in f:
                    self.scenes_json.append(json.loads(line))
            
            # Print scene statistics
            total_objects = sum(len(scene['objects']) for scene in self.scenes_json)
            avg_objects = total_objects / len(self.scenes_json)
            print(f"Total scenes: {len(self.scenes_json)}")
            print(f"Total objects: {total_objects}")
            print(f"Average objects per scene: {avg_objects:.2f}")
    
    def _extract_question_type(self, question):
        """Extract question type from program safely"""
        if 'program' not in question or not question['program']:
            return 'unknown'
        
        last_node = question['program'][-1]
        if isinstance(last_node, dict) and 'type' in last_node:
            return last_node['type']
        return 'unknown'
    
    def analyze_dataset_overview(self):
        """Print dataset overview statistics"""
        print("\n" + "=" * 80)
        print("DATASET OVERVIEW")
        print("=" * 80)
        
        print(f"\nSplit: {self.split}")
        print(f"Total questions: {len(self.dataset)}")
        print(f"Total images: {len(set(q['image_id'] for q in self.dataset.questions))}")
        print(f"Answer vocabulary size: {len(self.dataset.answer_vocab)}")
        
        if self.scenes_json:
            total_objects = sum(len(scene['objects']) for scene in self.scenes_json)
            avg_objects = total_objects / len(self.scenes_json)
            print(f"Total scenes: {len(self.scenes_json)}")
            print(f"Total objects: {total_objects}")
            print(f"Average objects per scene: {avg_objects:.2f}")
    
    def analyze_question_types(self):
        """Analyze question type distribution"""
        print("\n" + "=" * 80)
        print("QUESTION TYPE ANALYSIS")
        print("=" * 80)
        
        # Extract question types
        question_types = [self._extract_question_type(q) for q in self.dataset.questions]
        type_counts = Counter(question_types)
        
        print("\nQuestion type distribution:")
        for q_type, count in type_counts.most_common():
            percentage = (count / len(question_types)) * 100
            print(f"  {q_type:25s}: {count:6d} ({percentage:5.2f}%)")
        
        # Visualize
        plt.figure(figsize=(12, 6))
        types, counts = zip(*type_counts.most_common())
        plt.bar(types, counts, color='steelblue')
        plt.xlabel('Question Type', fontsize=12)
        plt.ylabel('Count', fontsize=12)
        plt.title(f'Question Type Distribution ({self.split} split)', fontsize=14)
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.savefig(f'outputs/question_types_{self.split}.png', dpi=300, bbox_inches='tight')
        print(f"\n✓ Saved plot: outputs/question_types_{self.split}.png")
        plt.close()
    
    def analyze_program_complexity(self):
        """Analyze question program complexity"""
        print("\n" + "=" * 80)
        print("PROGRAM COMPLEXITY ANALYSIS")
        print("=" * 80)
        
        program_lengths = []
        for q in self.dataset.questions:
            if 'program' in q and q['program']:
                program_lengths.append(len(q['program']))
        
        if not program_lengths:
            print("No program data available")
            return
        
        print(f"\nProgram length statistics:")
        print(f"  Mean: {np.mean(program_lengths):.2f}")
        print(f"  Median: {np.median(program_lengths):.2f}")
        print(f"  Min: {min(program_lengths)}")
        print(f"  Max: {max(program_lengths)}")
        print(f"  Std: {np.std(program_lengths):.2f}")
        
        # Distribution
        length_counts = Counter(program_lengths)
        print("\nProgram length distribution:")
        for length in sorted(length_counts.keys()):
            count = length_counts[length]
            percentage = (count / len(program_lengths)) * 100
            print(f"  Length {length:2d}: {count:6d} ({percentage:5.2f}%)")
        
        # Visualize
        plt.figure(figsize=(10, 6))
        plt.hist(program_lengths, bins=range(1, max(program_lengths)+2), 
                 color='coral', edgecolor='black', alpha=0.7)
        plt.xlabel('Program Length', fontsize=12)
        plt.ylabel('Frequency', fontsize=12)
        plt.title(f'Program Length Distribution ({self.split} split)', fontsize=14)
        plt.tight_layout()
        plt.savefig(f'outputs/program_lengths_{self.split}.png', dpi=300, bbox_inches='tight')
        print(f"\n✓ Saved plot: outputs/program_lengths_{self.split}.png")
        plt.close()
    
    def analyze_answers(self):
        """Analyze answer distribution"""
        print("\n" + "=" * 80)
        print("ANSWER ANALYSIS")
        print("=" * 80)
        
        answers = [q['answer'] for q in self.dataset.questions if 'answer' in q]
        answer_counts = Counter(answers)
        
        print(f"\nTotal unique answers: {len(answer_counts)}")
        print(f"Most common answers:")
        for answer, count in answer_counts.most_common(20):
            percentage = (count / len(answers)) * 100
            print(f"  {answer:20s}: {count:6d} ({percentage:5.2f}%)")
        
        # Categorize answers
        yes_no = sum(1 for a in answers if a in ['yes', 'no'])
        numbers = sum(1 for a in answers if a.isdigit())
        colors = sum(1 for a in answers if a in ['red', 'blue', 'green', 'brown', 
                                                   'cyan', 'purple', 'yellow', 'gray'])
        shapes = sum(1 for a in answers if a in ['cube', 'sphere', 'cylinder'])
        materials = sum(1 for a in answers if a in ['metal', 'rubber'])
        sizes = sum(1 for a in answers if a in ['small', 'large'])
        
        print(f"\nAnswer categories:")
        print(f"  Yes/No: {yes_no} ({yes_no/len(answers)*100:.2f}%)")
        print(f"  Numbers: {numbers} ({numbers/len(answers)*100:.2f}%)")
        print(f"  Colors: {colors} ({colors/len(answers)*100:.2f}%)")
        print(f"  Shapes: {shapes} ({shapes/len(answers)*100:.2f}%)")
        print(f"  Materials: {materials} ({materials/len(answers)*100:.2f}%)")
        print(f"  Sizes: {sizes} ({sizes/len(answers)*100:.2f}%)")
    
    def analyze_scene_properties(self):
        """Analyze scene and object properties"""
        if self.scenes_json is None:
            print("\n" + "=" * 80)
            print("SCENE PROPERTIES ANALYSIS")
            print("=" * 80)
            print("\nScene analysis not available for test split")
            return
        
        print("\n" + "=" * 80)
        print("SCENE PROPERTIES ANALYSIS")
        print("=" * 80)
        
        # Object count distribution
        object_counts = [len(scene['objects']) for scene in self.scenes_json]
        
        print(f"\nObjects per scene:")
        print(f"  Mean: {np.mean(object_counts):.2f}")
        print(f"  Median: {np.median(object_counts):.2f}")
        print(f"  Min: {min(object_counts)}")
        print(f"  Max: {max(object_counts)}")
        
        # Attribute distributions
        colors = []
        shapes = []
        materials = []
        sizes = []
        
        for scene in self.scenes_json:
            for obj in scene['objects']:
                colors.append(obj['color'])
                shapes.append(obj['shape'])
                materials.append(obj['material'])
                sizes.append(obj['size'])
        
        print(f"\nAttribute distributions:")
        
        print(f"\n  Colors: {len(set(colors))} unique")
        color_counts = Counter(colors)
        for color, count in color_counts.most_common():
            print(f"    {color:10s}: {count:6d}")
        
        print(f"\n  Shapes: {len(set(shapes))} unique")
        shape_counts = Counter(shapes)
        for shape, count in shape_counts.most_common():
            print(f"    {shape:10s}: {count:6d}")
        
        print(f"\n  Materials: {len(set(materials))} unique")
        material_counts = Counter(materials)
        for material, count in material_counts.most_common():
            print(f"    {material:10s}: {count:6d}")
        
        print(f"\n  Sizes: {len(set(sizes))} unique")
        size_counts = Counter(sizes)
        for size, count in size_counts.most_common():
            print(f"    {size:10s}: {count:6d}")
        
        # Visualize attribute distributions
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        
        # Colors
        c_labels, c_counts = zip(*color_counts.most_common())
        axes[0, 0].bar(range(len(c_labels)), c_counts, color='steelblue')
        axes[0, 0].set_xticks(range(len(c_labels)))
        axes[0, 0].set_xticklabels(c_labels, rotation=45)
        axes[0, 0].set_title('Color Distribution', fontsize=12)
        axes[0, 0].set_ylabel('Count')
        
        # Shapes
        s_labels, s_counts = zip(*shape_counts.most_common())
        axes[0, 1].bar(s_labels, s_counts, color='steelblue')
        axes[0, 1].set_title('Shape Distribution', fontsize=12)
        axes[0, 1].set_ylabel('Count')
        
        # Materials
        m_labels, m_counts = zip(*material_counts.most_common())
        axes[1, 0].bar(m_labels, m_counts, color='coral')
        axes[1, 0].set_title('Material Distribution', fontsize=12)
        axes[1, 0].set_ylabel('Count')
        
        # Sizes
        sz_labels, sz_counts = zip(*size_counts.most_common())
        axes[1, 1].bar(sz_labels, sz_counts, color='mediumseagreen')
        axes[1, 1].set_title('Size Distribution', fontsize=12)
        axes[1, 1].set_ylabel('Count')
        
        plt.tight_layout()
        plt.savefig(f'outputs/object_attributes_{self.split}.png', dpi=300, bbox_inches='tight')
        print(f"\n✓ Saved plot: outputs/object_attributes_{self.split}.png")
        plt.close()
    
    def visualize_sample_images(self, num_samples: int = 6):
        """Visualize sample images with annotations"""
        print("\n" + "=" * 80)
        print("SAMPLE IMAGES")
        print("=" * 80)
    
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        axes = axes.flatten()
    
        # Select diverse samples
        indices = np.linspace(0, len(self.dataset)-1, num_samples, dtype=int)
    
        for idx, ax in zip(indices, axes):
            try:
                # Dataset returns a dictionary
                item = self.dataset[idx]
                
                # Extract components
                image = item['image']  # PIL Image
                question = item['question']  # str
                answer = item.get('answer', 'N/A')  # str
                
                # Display image
                ax.imshow(image)
                ax.axis('off')
                
                # Truncate question if too long
                question_text = question[:60] + '...' if len(question) > 60 else question
                ax.set_title(f"Q: {question_text}\nA: {answer}", fontsize=9, pad=10)
                
            except Exception as e:
                print(f"Warning: Could not load sample {idx}: {e}")
                ax.axis('off')
                ax.text(0.5, 0.5, f'Error loading\nsample {idx}', 
                       ha='center', va='center')
    
        plt.tight_layout()
        plt.savefig(f'outputs/sample_images_{self.split}.png', dpi=300, bbox_inches='tight')
        print(f"\n✓ Saved plot: outputs/sample_images_{self.split}.png")
        plt.close()

    def analyze_question_answer_pairs(self):
        """Analyze question-answer pair patterns"""
        print("\n" + "=" * 80)
        print("QUESTION-ANSWER PAIR ANALYSIS")
        print("=" * 80)
        
        # Group by question type and answer
        type_answer_pairs = defaultdict(Counter)
        
        for q in self.dataset.questions:
            if 'answer' in q:
                q_type = self._extract_question_type(q)
                answer = q['answer']
                type_answer_pairs[q_type][answer] += 1
        
        print("\nTop answers per question type:")
        for q_type in sorted(type_answer_pairs.keys()):
            print(f"\n  {q_type}:")
            for answer, count in type_answer_pairs[q_type].most_common(5):
                total = sum(type_answer_pairs[q_type].values())
                percentage = (count / total) * 100
                print(f"    {answer:15s}: {count:5d} ({percentage:5.2f}%)")
    
    def generate_summary_report(self):
        """Generate a comprehensive summary report"""
        print("\n" + "=" * 80)
        print("SUMMARY REPORT")
        print("=" * 80)
        
        report = {
            'split': self.split,
            'total_questions': len(self.dataset),
            'total_images': len(set(q['image_id'] for q in self.dataset.questions)),
            'answer_vocab_size': len(self.dataset.answer_vocab),
        }
        
        # Question types
        question_types = [self._extract_question_type(q) for q in self.dataset.questions]
        report['question_types'] = dict(Counter(question_types).most_common())
        
        # Program lengths
        program_lengths = [len(q['program']) for q in self.dataset.questions 
                          if 'program' in q and q['program']]
        if program_lengths:
            report['avg_program_length'] = float(np.mean(program_lengths))
            report['max_program_length'] = int(max(program_lengths))
        
        if self.scenes_json:
            object_counts = [len(scene['objects']) for scene in self.scenes_json]
            report['avg_objects_per_scene'] = float(np.mean(object_counts))
            report['total_objects'] = sum(object_counts)
        
        # Save report
        output_path = f'outputs/summary_report_{self.split}.json'
        with open(output_path, 'w') as f:
            json.dump(report, f, indent=2)
        
        print(f"\n✓ Saved summary report: {output_path}")
        
        # Print key findings
        print("\nKey Findings:")
        print(f"  • Dataset contains {report['total_questions']} questions across {report['total_images']} images")
        print(f"  • {len(report['question_types'])} different question types")
        if 'avg_program_length' in report:
            print(f"  • Average program length: {report['avg_program_length']:.2f} operations")
        if 'avg_objects_per_scene' in report:
            print(f"  • Average {report['avg_objects_per_scene']:.2f} objects per scene")
    
    def run_full_analysis(self):
        """Run complete EDA pipeline"""
        print("\n" + "=" * 80)
        print(f"CLEVR DATASET - EXPLORATORY DATA ANALYSIS")
        print(f"Split: {self.split}")
        print("=" * 80)
        
        # Create output directory
        Path('outputs').mkdir(exist_ok=True)
        
        # Run all analyses
        try:
            self.analyze_dataset_overview()
            self.analyze_question_types()
            self.analyze_program_complexity()
            self.analyze_answers()
            self.analyze_scene_properties()
            self.visualize_sample_images()
            self.analyze_question_answer_pairs()
            self.generate_summary_report()
        except Exception as e:
            print(f"\n✗ Error during analysis: {e}")
            import traceback
            traceback.print_exc()
            return
        
        print("\n" + "=" * 80)
        print("✓ EDA COMPLETE!")
        print("=" * 80)
        print("\nGenerated files:")
        print(f"  • outputs/question_types_{self.split}.png")
        print(f"  • outputs/program_lengths_{self.split}.png")
        print(f"  • outputs/object_attributes_{self.split}.png")
        print(f"  • outputs/sample_images_{self.split}.png")
        print(f"  • outputs/summary_report_{self.split}.json")


def main():
    """Main execution function"""
    
    # Configuration
    DATA_DIR = "/home/nidszxh/Neuro-Symbolic-Visual-Reasoning-System/data/processed"
    
    if not Path(DATA_DIR).exists():
        print(f"Error: Dataset not found at {DATA_DIR}")
        print("Please ensure the processed dataset exists at the specified path")
        return
    
    # Run EDA on validation split
    print("Running EDA on validation split...")
    analyzer = CLEVREDAAnalyzer(DATA_DIR, split='val')
    analyzer.run_full_analysis()
    
    # Optionally run on train split
    run_train = input("\nRun EDA on training split? (y/n): ").lower().strip()
    if run_train == 'y':
        print("\nRunning EDA on training split...")
        analyzer_train = CLEVREDAAnalyzer(DATA_DIR, split='train')
        analyzer_train.run_full_analysis()


if __name__ == "__main__":
    main()

Running EDA on validation split...
Loading val split...
Loading processed questions from /home/nidszxh/Neuro-Symbolic-Visual-Reasoning-System/data/processed/processed_val_questions.jsonl...
Loading processed scenes from /home/nidszxh/Neuro-Symbolic-Visual-Reasoning-System/data/processed/processed_val_scenes.jsonl...
Loaded 149991 questions for 'val' split
Answer vocabulary size: 28
Total scenes: 15000
Total objects: 97358
Average objects per scene: 6.49

CLEVR DATASET - EXPLORATORY DATA ANALYSIS
Split: val

DATASET OVERVIEW

Split: val
Total questions: 149991
Total images: 15000
Answer vocabulary size: 28
Total scenes: 15000
Total objects: 97358
Average objects per scene: 6.49

QUESTION TYPE ANALYSIS

Question type distribution:
  unknown                  : 149991 (100.00%)

✓ Saved plot: outputs/question_types_val.png

PROGRAM COMPLEXITY ANALYSIS

Program length statistics:
  Mean: 11.16
  Median: 10.00
  Min: 2
  Max: 25
  Std: 4.29

Program length distribution:
  Length  2:     44 (